# Gold v2 — 모순 재질문 agent (consistency re-ask)

오프라인 분석에서 normal reference 답의 Q1·Q2 모순이 많았습니다 (LLaVA 59%, Qwen 34%, InternVL 22%, MiniCPM 13%).
이 노트북은 **모순(또는 PatchCore 충돌)이 난 이미지만** 모델에게 실제로 다시 묻습니다.

- 1단계: 기존 normal reference 예측 재사용 (추론 없음)
- 2단계 (라우팅된 이미지만, Q1·Q2 각각 1회): `[정상 reference, target]` + "1차 답은 Q1 '…', Q2 '…'였고 서로 모순된다, 다시 확인하라"
  - `self`: Q1·Q2 모순 이미지만, 점수 문장 없음
  - `self_score`: 모순 또는 PatchCore 강한 충돌(10/90, 고정) 이미지, PatchCore 점수 문장 추가
- Q3: `R_*`는 1단계 그대로, `R_*+S`는 원본만 답 (사후 규칙)
- 비교: 고정 4조건, 오프라인 A5, 이미지 단위 paired bootstrap 95% CI

GPU: **L4** (4bit 금지). 예상: 아래 '라우팅 미리보기' 셀이 모델별 호출 수와 시간을 출력합니다 (대략 총 1시간 안팎, LLaVA가 가장 많음).
20문항마다 Drive에 저장되고, 끊기면 위에서부터 다시 실행하면 이어서 돕니다.

결과: `MyDrive/lvlm_insplad_results/gold_v2_consistency_agent/` → GitHub `results/latest/gold_v2_consistency_agent/`

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
NORMAL_REF_FOLDER = "lvlm_insplad_results/gold_reference_v2/normal_reference/predictions"
EXPERT_FOLDER = "lvlm_insplad_results/gold_v2_vision_expert_full/eval/predictions"
COMBINED_FOLDER = "lvlm_insplad_results/gold_v2_reference_expert/predictions"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_consistency_agent"

MODELS = ["qwen3_5_4b","internvl3_5_8b","minicpm_v_4_5","llava_onevision_7b"]
VARIANTS = ["self","self_score"]
EXPERT_PAIR = "patchcore__score"
CONFLICT_LOW, CONFLICT_HIGH = 10.0, 90.0   # 고정 (사후 튜닝 금지)
CHECKPOINT_EVERY = 20
PUBLISH = True

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""   # 비우면 자동 검색
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색

In [ ]:
import base64, csv, gc, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
NORMAL_REF_DIR = DRIVE_ROOT / NORMAL_REF_FOLDER
EXPERT_DIR = DRIVE_ROOT / EXPERT_FOLDER
COMBINED_DIR = DRIVE_ROOT / COMBINED_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""))
    return subprocess.run(cmd, env=env, check=check)

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT":"1",
    "GIT_CONFIG_KEY_0":"http.extraHeader",
    "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}",
    "GIT_TERMINAL_PROMPT":"0",
})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env

# LVLM 추론만 하므로 anomalib 등 expert 패키지는 설치하지 않습니다.
run([sys.executable,"-m","pip","install","-q",
     "transformers==5.14.0","accelerate==1.14.0","sentencepiece==0.2.2",
     "pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))

import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")

In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음). 경로를 알면 설정 셀의 *_ZIP_PATH에 넣으세요.", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

def read_jsonl(path):
    return [json.loads(x) for x in Path(path).read_text(encoding="utf-8").splitlines() if x.strip()]

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP)
print("UNSUP:", UNSUP_ZIP)


IMAGE_ONLY_CSV = REPO_DIR / "results/latest/gold_v2_baseline/prediction_review.csv"
need = [IMAGE_ONLY_CSV]
for m in MODELS:
    need += [NORMAL_REF_DIR / f"{m}.jsonl", EXPERT_DIR / m / f"{EXPERT_PAIR}.jsonl", COMBINED_DIR / m / f"{EXPERT_PAIR}.jsonl"]
missing = [str(p) for p in need if not p.is_file()]
if missing:
    for p in missing:
        print("없음", p)
    raise FileNotFoundError(f"{len(missing)}개 입력 없음")
print("입력 예측 파일", len(need), "개 확인")

In [ ]:
# 라우팅 미리보기 — 모델별로 다시 물을 이미지 수와 예상 시간 (추론 없음)
import run_gold_v2_offline_agent as oa
import run_gold_v2_baselines as v2_runner
rows = v2_runner.load_gold_qa_v2(QA_DIR)
preds, pct = oa.load_predictions(MODELS, IMAGE_ONLY_CSV, NORMAL_REF_DIR, EXPERT_DIR, COMBINED_DIR, EXPERT_PAIR)
oa.check_coverage(rows, preds, pct)
SEC_PER_CALL = {"qwen3_5_4b": 1.5, "internvl3_5_8b": 2.0, "minicpm_v_4_5": 2.5, "llava_onevision_7b": 4.0}  # 대략값
plan = []
for m in MODELS:
    conflict, inconsistent = oa.routing_sets(rows, preds[(m, "normal_ref")], pct, CONFLICT_LOW, CONFLICT_HIGH)
    for v in VARIANTS:
        routed = inconsistent if v == "self" else inconsistent | conflict
        plan.append({"model": m, "variant": v, "routed_images": len(routed), "calls": 2 * len(routed),
                     "approx_min": round(2 * len(routed) * SEC_PER_CALL[m] / 60, 1)})
plan = pd.DataFrame(plan)
display(plan)
print(f"총 호출 {plan.calls.sum()}회, 대략 {plan.approx_min.sum():.0f}분 + 모델 로딩")

In [ ]:
# 2단계 재질문 실행 — 끊기면 이 셀만 다시 실행해도 완료된 문항은 재사용
run([
    sys.executable, REPO_DIR / "scripts/run_gold_v2_consistency_agent.py",
    "--qa-dir", QA_DIR,
    "--sup-zip", SUP_ZIP,
    "--unsup-zip", UNSUP_ZIP,
    "--image-only-csv", IMAGE_ONLY_CSV,
    "--normal-ref-dir", NORMAL_REF_DIR,
    "--expert-dir", EXPERT_DIR,
    "--combined-dir", COMBINED_DIR,
    "--out-dir", RUN_ROOT,
    "--models", *MODELS,
    "--variants", *VARIANTS,
    "--expert-pair", EXPERT_PAIR,
    "--conflict-low", str(CONFLICT_LOW),
    "--conflict-high", str(CONFLICT_HIGH),
    "--checkpoint-every", str(CHECKPOINT_EVERY),
])

In [ ]:
# 결과
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 200)
lb = pd.read_csv(RUN_ROOT / "leaderboard.csv")
show = ["model_key","policy","source","Q1_anomaly_f1","Q2_defect_macro_f1","Q3_loc_acc",
        "d_Q1_vs_normal_ref","d_Q2_vs_normal_ref","d_Q3_vs_normal_ref",
        "all3_at_least_best_fixed","avg_lvlm_calls_per_q"]
display(lb[show].round(3))
sig = pd.read_csv(RUN_ROOT / "significance.csv")
for q in ("Q1", "Q2", "Q3"):
    sig[f"{q}_sig"] = sig[f"{q}_ci_low"] > 0
print("\n[유의성] CI 하한 > 0 이면 우연이 아닌 개선. S_q3_image_only 대비 = agent 자체 효과, A5 대비 = 실제 재질문이 오프라인 결합보다 나은지")
display(sig[["model_key","baseline","policy","Q1_delta","Q1_ci_low","Q1_ci_high","Q1_sig",
             "Q2_delta","Q2_ci_low","Q2_ci_high","Q2_sig","Q3_delta","Q3_sig"]].round(3))
print((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))

In [ ]:
# GitHub publish
if PUBLISH:
    publish_files = ["leaderboard.csv", "significance.csv", "run_status.json", "run_metadata.json"]
    run([
        sys.executable, REPO_DIR / "scripts/publish_results_to_github.py",
        "--out-dir", RUN_ROOT,
        "--repo", REPO,
        "--branch", REPO_REF,
        "--prefix", "gold_v2_consistency_agent",
        "--files", *[f for f in publish_files if (RUN_ROOT / f).is_file()],
    ], env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_consistency_agent/")

## 완료 후

- `run_status.json`의 `row_errors`가 모두 0인지 확인 (0이 아니면 실행 셀만 다시 실행 → 실패 문항만 재시도)
- 핵심: `R_self+S` / `R_self_score+S`가 `all3_at_least_best_fixed`이고, `S_q3_image_only` 대비 Q1·Q2 CI 하한 > 0 인지
- 재질문 프롬프트에 1차 답을 그대로 보여주므로, 모델이 단순히 답을 바꾸는 경향(번복 편향)이 있을 수 있음 — `predictions/*/*.jsonl`의 `first_choice` vs `predicted_choice`로 번복률 확인 가능